# nb50 - Clean window scan (Act 1 of the 0.0235 campaign)

**Error analysis.** Gap decomposition 2026-08-01: pileup penalty is ~0.003 (minbias 0.0426 vs clean 0.0397) while the clean-side gap to the metric floor is ~0.016 (clean 0.0397 vs floor 0.0235). On clean data the residual correlates 0.977 with in-window containment (nb16) - the shower tail lost outside the 9x9 window is the dominant clean-side error.

**Question.** Does a wider window recover the containment term on clean data, where widening has NO pileup cost?

**Hypothesis.** Act-1/H13: W=6 (13x13) or W=8 (17x17) beats W=4 on the clean sample with the current stack; the residual-containment correlation should shrink as W grows if the mechanism is real.

**Research.** Window scans were only ever run on minbias (nb32, pre-quant, W=4 optimum - a pileup trade-off, not a containment statement). CRILIN software compensation shows shape observables regress the contained fraction (arXiv:2606.05111); the clean-side scan isolates the containment lever from pileup entirely.

**Proof criterion.** Same notebook, same splits, 2 seeds per W; win = any W beats clean-W4 by >0.002 overall. Anchors: clean best 0.0397 (nb19 MeanResidual); metric floor on the clean spectrum computed in the verdict cell.

**GPU caution.** W=6 -> 169 tokens, W=8 -> 289 tokens - beyond the proven-safe 81-token class on this laptop. Batch is scaled down (96/48/24) and jobs run in order W=4 -> 6 -> 8; fall back to CPU if instability appears.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
from picocal_data import build_grid, make_windows, splits_for, THRESH, NC
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB50_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB50_MODE', 'full')
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': CLF = CLF[:4]
t0 = time.time()
CE = build_grid(CLF, 'clean')
print(f'device {DEVICE} | mode {MODE} | build {time.time()-t0:.0f}s')

clean: 30303 events
device cuda | mode full | build 41s


In [2]:
NG = 5
def prep_w(W):
    rows, keep = make_windows(W, CE)
    ktr, kva, kte = splits_for(keep, len(CE))
    N = len(rows); L = (2*W+1)**2; IN_DIM = rows[0][0].shape[1]
    y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
    Et = np.array([r[3] for r in rows], np.float32)
    sumE = np.array([r[1] for r in rows], np.float32)
    cont_ev = np.clip(sumE / (1000.0 * Et), 0, 2)
    X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
    G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
    for i, (tok, se, sde, et, rg, etv) in enumerate(rows):
        n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
        e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
        lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
        fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
        G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat]
    la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
    G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
    cont = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
    mean = cont.mean(0); std = cont.std(0) + EPS
    X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
    T = dict(X=torch.from_numpy(X).to(DEVICE), M=torch.from_numpy(M).to(DEVICE),
             G=torch.from_numpy(G).to(DEVICE), Y=torch.from_numpy(y).unsqueeze(1).to(DEVICE),
             E=torch.from_numpy(Eraw).to(DEVICE))
    print(f'W={W}: N {N}, tr/va/te {len(ktr)}/{len(kva)}/{len(kte)}, L {L}, IN_DIM {IN_DIM}')
    return dict(T=T, y=y, Et=Et, cont=cont_ev, ktr=ktr, kva=kva, kte=kte,
                IN_DIM=IN_DIM, la0=float(la0), lb0=float(lb0))

In [3]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4)
BATCH = {4: 96, 6: 48, 8: 24}
class SubNetC(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = torch.sigmoid(self.fhead(h).squeeze(-1)) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
QS = torch.tensor([0.25, 0.5, 0.75], device=DEVICE)
def pinball(q, yb):
    d = yb - q
    return torch.maximum(QS * d, (QS - 1) * d).mean()
def wcalib(qv, qt, yva):
    wv = qv[:, 2] - qv[:, 0]; wt_ = qt[:, 2] - qt[:, 0]
    cuts = np.quantile(wv, [1/3, 2/3])
    gv = np.digitize(wv, cuts); gt = np.digitize(wt_, cuts)
    pe = np.empty(len(qt))
    for g in range(3):
        if (gv == g).sum() < 10 or (gt == g).sum() == 0:
            a, b2 = np.polyfit(qv[:, 1], yva, 1)
        else:
            a, b2 = np.polyfit(qv[gv == g, 1], yva[gv == g], 1)
        pe[gt == g] = np.exp(a * qt[gt == g, 1] + b2)
    return pe
def train_eval(P, W, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetC(P['IN_DIM'], P['la0'], P['lb0']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ck = CKPT / f'nb50_W{W}_s{seed}.pt'
    T = P['T']
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(b): return model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def run(idx):
        model.eval(); out = []
        with torch.no_grad():
            for b in batches(idx, 128, False): out.append(fwd(b).cpu().numpy())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(P['kva'], 128, False):
                s += pinball(fwd(b), T['Y'][b]).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume W{W} s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(P['ktr'], BATCH[W], True):
            opt.zero_grad()
            pinball(fwd(b), T['Y'][b]).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    pe = wcalib(run(P['kva']), run(P['kte']), P['y'][P['kva']])
    return float(resolution(pe, P['Et'][P['kte']])['sigma_eff']), pe

In [4]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
JOBS = {'smoke': [(4, 0), (6, 0)],
        'full': [(W, s) for W in (4, 6, 8) for s in (0, 1)]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb50_clean_wscan{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['W'], prev['seed']))
    print('resume, done:', sorted(done))
PREPS = {}
for W, seed in JOBS:
    if (W, seed) in done: print('skip', W, seed); continue
    if W not in PREPS: PREPS[W] = prep_w(W)
    t1 = time.time()
    sig, pe = train_eval(PREPS[W], W, seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb50_pred{TAG}_W{W}_s{seed}.npy', pe)
    row = dict(W=W, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'W{W} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

W=4: N 30303, tr/va/te 21212/4545/4546, L 81, IN_DIM 16


W4 seed 0: sigma_eff 0.0498 (215s)


W4 seed 1: sigma_eff 0.0513 (274s)


W=6: N 30303, tr/va/te 21212/4545/4546, L 169, IN_DIM 16


W6 seed 0: sigma_eff 0.0555 (445s)


W6 seed 1: sigma_eff 0.0502 (457s)


W=8: N 30303, tr/va/te 21212/4545/4546, L 289, IN_DIM 16


W8 seed 0: sigma_eff 0.0509 (1166s)


W8 seed 1: sigma_eff 0.0491 (1088s)


 W  seed  sigma_eff  elapsed
 4     0     0.0498      215
 4     1     0.0513      274
 6     0     0.0555      445
 6     1     0.0502      457
 8     0     0.0509     1166
 8     1     0.0491     1088


## Verdict

Win = any W beats W=4 by >0.002 overall (2-seed means). Mechanism check: corr(residual, containment) must SHRINK with W if widening truly eats the containment term. Per-bin table against the metric floor at bin medians.

In [5]:
for W in (4, 6, 8):
    if W not in PREPS:
        try: PREPS[W] = prep_w(W)
        except Exception: continue
print('anchors: clean best 0.0397 (nb19 MeanResidual) | aggregate floor 0.0235 (10%/sqrtE + 1%)')
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
for W in (4, 6, 8):
    if W not in PREPS: continue
    P = PREPS[W]
    preds = [np.load(OUT / f'nb50_pred{TAG}_W{W}_s{s}.npy') for s in SEEDS
             if (OUT / f'nb50_pred{TAG}_W{W}_s{s}.npy').exists()]
    if not preds: continue
    te_e = P['Et'][P['kte']]; cont_te = P['cont'][P['kte']]
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    r = (ens - te_e) / te_e
    cc = np.corrcoef(r, cont_te)[0, 1]
    edges = np.quantile(te_e, np.linspace(0, 1, 7))
    bins = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        floor = np.sqrt((0.10 / np.sqrt(np.median(te_e[mm]))) ** 2 + 0.01 ** 2)
        bins.append(f'{resolution(ens[mm], te_e[mm])["sigma_eff"]:.4f}(fl {floor:.3f})')
    print(f'W={W}: mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f} | corr(r, containment) {cc:+.3f}')
    print('   per-bin ' + ' / '.join(bins))

anchors: clean best 0.0397 (nb19 MeanResidual) | aggregate floor 0.0235 (10%/sqrtE + 1%)
W=4: mean 0.0505 +/- 0.0008 | ens 0.0491 | corr(r, containment) +0.569
   per-bin 0.0487(fl 0.039) / 0.0325(fl 0.029) / 0.0233(fl 0.024) / 0.0230(fl 0.021) / 0.0202(fl 0.018) / 0.0286(fl 0.016)
W=6: mean 0.0529 +/- 0.0026 | ens 0.0523 | corr(r, containment) +0.604
   per-bin 0.0456(fl 0.039) / 0.0308(fl 0.029) / 0.0247(fl 0.024) / 0.0234(fl 0.021) / 0.0210(fl 0.018) / 0.0301(fl 0.016)


W=8: mean 0.0500 +/- 0.0009 | ens 0.0500 | corr(r, containment) +0.553
   per-bin 0.0499(fl 0.039) / 0.0318(fl 0.029) / 0.0246(fl 0.024) / 0.0228(fl 0.021) / 0.0204(fl 0.018) / 0.0292(fl 0.016)
